# CMS hospital data

Sources: [dataset versions](https://data.cms.gov/provider-summary-by-type-of-service/medicare-inpatient-hospitals/medicare-inpatient-hospitals-by-provider-and-service/api-docs), [CMS API pagination](https://data.cms.gov/api-docs). CMS's GET allows at most 5,000 rows per page.


| Included features | Type |
|---|---|
| `Rndrng_Prvdr_CCN` | TEXT |
| `Rndrng_Prvdr_Org_Name` | TEXT |
| `Rndrng_Prvdr_City` | TEXT |
| `Rndrng_Prvdr_St` | TEXT |
| `Rndrng_Prvdr_State_FIPS` | TEXT |
| `Rndrng_Prvdr_Zip5` | TEXT |
| `Rndrng_Prvdr_State_Abrvtn` | TEXT |
| `Rndrng_Prvdr_RUCA` | TEXT |
| `Rndrng_Prvdr_RUCA_Desc` | TEXT |
| `DRG_Cd` | TEXT |
| `DRG_Desc` | TEXT |
| `Tot_Dschrgs` | NUMERIC |
| `Avg_Submtd_Cvrd_Chrg` | NUMERIC |
| `Avg_Tot_Pymt_Amt` | NUMERIC |
| `Avg_Mdcr_Pymt_Amt` | NUMERIC |

### Get data sets from CMS

In [18]:
from pathlib import Path
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

ROOT = Path.cwd()  # Run this notebook from the project folder.
DATASETS = {
    2024: "https://data.cms.gov/data-api/v1/dataset/ca9b5ef0-1386-4759-b2b6-5f9b35116786/data",
    2023: "https://data.cms.gov/data-api/v1/dataset/2f1e57ea-ac2f-4f62-aeb1-f8254307c395/data",
}
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=4, backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504], allowed_methods=["GET"],
)))

def get_page(url, size=5000, offset=0, filters=None):
    if not 1 <= size <= 5000 or offset < 0:
        raise ValueError("Use size 1..5000 and a nonnegative offset.")
    params = dict(filters or {})
    params.update({"size": size, "offset": offset})
    response = session.get(url, params=params, timeout=(15, 120))
    response.raise_for_status()
    records = response.json()
    if not isinstance(records, list):
        raise ValueError("Expected a JSON list of records.")
    return records


def search_column(url, column_name, column_value):
    """Return all rows matching a column value exactly, or [] if none match.

    Pass identifiers as strings to preserve leading zeros (e.g. "010001").
    """
    if not isinstance(column_name, str) or not column_name.strip():
        raise ValueError("column_name must be a nonempty string.")
    if column_value is None:
        raise ValueError("column_value must not be None.")
    filters = {f"filter[{column_name}]": str(column_value)}
    records = []
    while True:
        page = get_page(url, size=5000, offset=len(records), filters=filters)
        if not page:
            return records
        records.extend(page)


def get_all(url):
    """Return every record, paging until the API returns no more rows."""
    records = []
    while True:
        page = get_page(url, offset=len(records))
        if not page:
            return records
        records.extend(page)
        print(f"Downloaded {len(records):,} rows", flush=True)


def to_frame(records, year):
    """Preserve identifier strings and convert measures to numeric columns."""
    frame = pd.DataFrame(records)
    if frame.empty:
        raise ValueError(f"No records returned for {year}.")
    for column in ["Rndrng_Prvdr_CCN", "Rndrng_Prvdr_Zip5", "DRG_Cd",
                   "Rndrng_Prvdr_State_FIPS", "Rndrng_Prvdr_RUCA"]:
        frame[column] = frame[column].astype("string")
    for column in ["Tot_Dschrgs", "Avg_Submtd_Cvrd_Chrg",
                   "Avg_Tot_Pymt_Amt", "Avg_Mdcr_Pymt_Amt"]:
        frame[column] = pd.to_numeric(frame[column], errors="raise")
    frame["data_year"] = year
    return frame



def load_hsa_mapping():
    """Load and validate ZIP-to-HSA classifications once per workflow run."""
    crosswalk = pd.read_csv(
        ROOT / "HSA_Bdry__AK_HI_unmodified" / "ZipHsaHrr19.csv",
        dtype={"zipcode19": "string", "hsanum": "Int64"},
        usecols=["zipcode19", "hsanum"],
    ).rename(columns={"zipcode19": "_zip5", "hsanum": "HSA93"})
    crosswalk["_zip5"] = crosswalk["_zip5"].str.strip().str.zfill(5)
    if crosswalk["_zip5"].isna().any() or crosswalk["HSA93"].isna().any():
        raise ValueError("Crosswalk contains missing ZIP or HSA IDs.")
    crosswalk = crosswalk.drop_duplicates()
    if crosswalk["_zip5"].duplicated().any():
        raise ValueError("Crosswalk assigns a ZIP to multiple HSAs.")

    classifications = pd.read_csv(
        ROOT / "hsa_classifications.csv",
        dtype={"HSA93": "Int64", "classification": "Int64"},
    ).rename(columns={"classification": "hsa_classification"})
    if classifications["HSA93"].isna().any() or classifications["HSA93"].duplicated().any():
        raise ValueError("Classification HSA IDs must be present and unique.")
    if not classifications["hsa_classification"].isin([-1, 0, 1]).all():
        raise ValueError("HSA classifications must be -1, 0, or 1.")

    return crosswalk.merge(classifications, on="HSA93", how="left", validate="many_to_one")


def classify_hospitals(frame, hsa_mapping):
    """Keep significant HSAs using a preloaded mapping; ai_class is 1 or 0."""
    # Drop generated columns so this function can be re-run on its own output.
    hospitals = frame.drop(
        columns=["HSA93", "hsa_id", "hsa_classification", "hotspot", "ai_class", "_zip5"], errors="ignore"
    ).copy()
    # Accept five-digit ZIPs, ZIP+4, or short digit strings missing leading zeros.
    zip_text = hospitals["Rndrng_Prvdr_Zip5"].astype("string").str.strip()
    hospitals["_zip5"] = zip_text.str.extract(
        r"^(\d{1,5})(?:-\d{4})?$", expand=False
    ).str.zfill(5)
    mapped = hospitals.merge(hsa_mapping, on="_zip5", how="left", validate="many_to_one")
    keep = mapped["hsa_classification"].isin([1, -1])
    significant = mapped.loc[keep].drop(columns="_zip5").copy()
    significant["ai_class"] = significant["hsa_classification"].map({1: 1, -1: 0}).astype("int64")
    significant = significant.rename(columns={"HSA93": "hsa_id"}).reset_index(drop=True)

    print(f"Input rows: {len(frame):,}")
    print(f"Excluded insignificant: {mapped['hsa_classification'].eq(0).sum():,}")
    print(f"Excluded unmatched ZIP/HSA: {mapped['hsa_classification'].isna().sum():,}")
    print(f"Kept hotspots: {significant['ai_class'].eq(1).sum():,}")
    print(f"Kept coldspots: {significant['ai_class'].eq(0).sum():,}")
    return significant


### Fetch, classify, and save significant records
Download both complete datasets, map hospital ZIPs through `ZipHsaHrr19.csv` to `hsa_id`, then join `hsa_classifications.csv`.

Only HSA classifications **1** (hotspot) and **-1** (coldspot) are saved to `get_data.csv`. Insignificant HSAs (**0**) and unmatched ZIP/HSA rows are excluded. The new binary `ai_class` column is **1 for hotspot, 0 for coldspot**. `hsa_id` and `hsa_classification` preserve the mapping and original label. All hospital/DRG rows for qualifying hospitals are retained, with `data_year` distinguishing years.

The mapping files are loaded once per run. Each year is downloaded and filtered separately; only retained rows are stored in `datasets_by_year` and concatenated into `all_data`. Full unfiltered yearly datasets are released after filtering. The ZIP crosswalk is from 2019, applied to both data years. Re-run after changing the classifications to include any newly qualifying hospitals.

In [ ]:
hsa_mapping = load_hsa_mapping()
datasets_by_year = {}
for year, url in DATASETS.items():
    print(f"Processing {year}")
    datasets_by_year[year] = classify_hospitals(
        to_frame(get_all(url), year), hsa_mapping
    )
all_data = pd.concat(datasets_by_year.values(), ignore_index=True)
assert len(all_data) == sum(len(frame) for frame in datasets_by_year.values())
all_data.to_csv(ROOT / "hospital_data.csv", index=False)
print(f"Saved {len(all_data):,} significant rows to hospital_data.csv")
all_data.groupby(["data_year", "ai_class"]).size().rename("rows").to_frame()

Processing 2024
Downloaded 5,000 rows
Downloaded 10,000 rows
Downloaded 15,000 rows
Downloaded 20,000 rows
Downloaded 25,000 rows
Downloaded 30,000 rows
Downloaded 35,000 rows
Downloaded 40,000 rows
Downloaded 45,000 rows
Downloaded 50,000 rows
Downloaded 55,000 rows
Downloaded 60,000 rows
Downloaded 65,000 rows
Downloaded 70,000 rows
Downloaded 75,000 rows
Downloaded 80,000 rows
Downloaded 85,000 rows
Downloaded 90,000 rows
Downloaded 95,000 rows
Downloaded 100,000 rows
Downloaded 105,000 rows
Downloaded 110,000 rows
Downloaded 115,000 rows
Downloaded 120,000 rows
Downloaded 125,000 rows
Downloaded 130,000 rows
Downloaded 135,000 rows
Downloaded 140,000 rows
Downloaded 145,000 rows
Downloaded 145,879 rows
Input rows: 145,879
Excluded insignificant: 116,097
Excluded unmatched ZIP/HSA: 0
Kept hotspots: 27,669
Kept coldspots: 2,113
Processing 2023
Downloaded 5,000 rows
Downloaded 10,000 rows
Downloaded 15,000 rows
Downloaded 20,000 rows
Downloaded 25,000 rows
Downloaded 30,000 rows
Downl

rows
data_year ai_class       
2023      0          2146
          1         27909
2024      0          2113
          1         27669

### Distinct hospitals by category
Counts use unique hospital CCNs across both years. A hospital appearing in both categories is counted once in each.

In [20]:
# Count each hospital CCN once per category across all years and DRGs.
hospital_counts = (
    all_data.groupby("ai_class")["Rndrng_Prvdr_CCN"]
    .nunique()
    .reindex([1, 0], fill_value=0)
)
print(f"Distinct hotspot hospitals (ai_class=1): {hospital_counts.loc[1]:,}")
print(f"Distinct coldspot hospitals (ai_class=0): {hospital_counts.loc[0]:,}")

Distinct hotspot hospitals (ai_class=1): 449
Distinct coldspot hospitals (ai_class=0): 58


### Search the API by column
Just an example


In [21]:
matching_records = search_column(
    DATASETS[2024], "Rndrng_Prvdr_Org_Name", "Southeast Health Medical Center"
)
matching_df = pd.DataFrame(matching_records)
print(matching_df)

   Rndrng_Prvdr_CCN            Rndrng_Prvdr_Org_Name Rndrng_Prvdr_City  \
0            010001  Southeast Health Medical Center            Dothan   
1            010001  Southeast Health Medical Center            Dothan   
2            010001  Southeast Health Medical Center            Dothan   
3            010001  Southeast Health Medical Center            Dothan   
4            010001  Southeast Health Medical Center            Dothan   
..              ...                              ...               ...   
90           010001  Southeast Health Medical Center            Dothan   
91           010001  Southeast Health Medical Center            Dothan   
92           010001  Southeast Health Medical Center            Dothan   
93           010001  Southeast Health Medical Center            Dothan   
94           010001  Southeast Health Medical Center            Dothan   

           Rndrng_Prvdr_St Rndrng_Prvdr_State_FIPS Rndrng_Prvdr_Zip5  \
0   1108 Ross Clark Circle             